# Notebook 04 — Stress Testing

Applies all six predefined historical scenarios to the SPY/QQQ/IWM portfolio.
Computes reverse stress test: "what shock causes a 20% portfolio loss?"
Compares stressed ES to normal-period ES.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from src.stress_testing import (
    historical_scenario_var, run_all_scenarios,
    reverse_stress_test, PREDEFINED_SCENARIOS
)
from src.expected_shortfall import historical_es, stressed_es

print('Libraries loaded.')

In [ ]:
data = yf.download(['SPY', 'QQQ', 'IWM'], start='2018-01-01', end='2024-01-01', progress=False)['Close']
returns = data.pct_change().dropna()
weights = np.array([1/3, 1/3, 1/3])
port_returns = (returns * weights).sum(axis=1)
portfolio_value = 1_000_000  # $1M portfolio

print(f'Portfolio returns: {len(port_returns)} observations')
print(f'Annual vol: {port_returns.std() * np.sqrt(252):.1%}')

## 1. All Predefined Scenarios

In [ ]:
scenario_df = run_all_scenarios(port_returns, portfolio_value)
print(scenario_df[['Scenario', 'Equity Shock', 'Portfolio Loss ($)', 'Normal VaR', 'Stressed VaR', 'VaR Ratio']].to_string(index=False))

In [ ]:
# Bar chart of scenario losses
scenario_names = list(PREDEFINED_SCENARIOS.keys())
losses = []
for name in scenario_names:
    r = historical_scenario_var(port_returns, name, portfolio_value)
    losses.append(abs(r['portfolio_loss']))

fig = go.Figure(go.Bar(
    x=scenario_names, y=losses,
    marker_color='crimson',
    text=[f'${l:,.0f}' for l in losses], textposition='auto'
))
fig.update_layout(
    title=f'Portfolio Loss ($1M) by Historical Scenario',
    xaxis_title='Scenario', yaxis_title='Estimated Loss ($)',
    height=450
)
fig.show()

## 2. Stressed ES vs Normal ES

In [ ]:
normal_es = historical_es(port_returns, 0.05)

stress_windows = [
    ('COVID-19', ('2020-02-01', '2020-06-01')),
    ('2022 Rate Shock', ('2022-01-01', '2022-12-31')),
]

print(f'Normal-period ES (95%):  {normal_es:.3%}')
for name, window in stress_windows:
    s_es = stressed_es(port_returns, window, 0.05)
    ratio = s_es / normal_es
    print(f'{name} Stressed ES: {s_es:.3%}  (ratio: {ratio:.2f}x normal)')

## 3. Reverse Stress Test — "What Breaks Us?"

In [ ]:
for target_pct in [0.10, 0.20, 0.30]:
    target_loss = portfolio_value * target_pct
    result = reverse_stress_test(port_returns, portfolio_value, target_loss)
    print(f'\nTarget loss: {target_pct:.0%} (${target_loss:,.0f})')
    print(f'  Implied equity shock:    {result["implied_equity_shock"]:.1%}')
    print(f'  Implied vol multiplier:  {result["implied_vol_multiplier"]:.1f}x')
    print(f'  Comparable scenario:     {result["comparable_scenario"]}')
    print(f'  Exceeds historical worst: {result["exceeds_historical_worst"]}')

In [ ]:
# Reverse stress test curve
target_losses = np.linspace(0.05, 0.50, 20)
implied_shocks = []
for tl in target_losses:
    r = reverse_stress_test(port_returns, portfolio_value, portfolio_value * tl)
    implied_shocks.append(r['implied_equity_shock'])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=target_losses * 100, y=[abs(s) * 100 for s in implied_shocks],
    mode='lines+markers', name='Required equity shock',
    line=dict(color='crimson', width=2)
))

# Mark known historical scenarios
for name, params in PREDEFINED_SCENARIOS.items():
    shock = abs(params['equity_shock'])
    loss  = shock  # simplified: loss ≈ shock for equity-only portfolio
    fig.add_trace(go.Scatter(
        x=[loss * 100], y=[shock * 100], mode='markers+text',
        text=[name.split('(')[0].strip()], textposition='top center',
        marker=dict(size=10, color='navy'), showlegend=False
    ))

fig.update_layout(
    title='Reverse Stress Test: Required Equity Shock to Achieve Target Loss',
    xaxis_title='Target Loss (% of portfolio)',
    yaxis_title='Required Equity Shock (%)',
    height=450
)
fig.show()